# 1. Import and Hardware Setup

In [66]:
import torch
print(torch.cuda.get_arch_list())
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import DataLoader, random_split, Subset
from torchvision import datasets, transforms

import matplotlib.pyplot as plt
import numpy as np
!pip install wandb -q
import wandb

!pip install tqdm ipywidgets -q
from tqdm.auto import tqdm

['sm_70', 'sm_75', 'sm_80', 'sm_86', 'sm_90', 'sm_100', 'sm_120']


In [67]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [68]:
DATA_PATH = "./Data"

In [69]:
wandb.login()

wandb: WARNING Calling wandb.login() after wandb.init() has no effect.


False

# 2. Hyperparameters

In [70]:
IMG_SIZE = 224
IN_CHANNELS = 3
BATCH_SIZE = 128
NUM_CLS = 101

LR = 0.01
WEIGHT_DECAY = 5e-4
DROPOUT_RATE = 0.4
EPOCHS = 100

SEED = 42

# 3. Data Preparation

In [71]:
# Calculated mean and std of Food101
stats = ((0.545, 0.443, 0.344), (0.269, 0.271, 0.276))

# Follow paper-style augmentation: resize -> random crop -> flip
train_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.RandomCrop(IMG_SIZE),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(*stats)
])

test_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(*stats)
])

In [72]:
import os
import random

def set_seed(seed: int = 42):
    os.environ['PYTHONHASHSEED'] = str(seed)
    
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = True
    
    try:
        torch.use_deterministic_algorithms(True, warn_only=True)
    except Exception:
        pass

set_seed(SEED)

In [73]:
# Download train data first without transform
dummy_data = datasets.Food101(root=DATA_PATH, split="train", download=True)

# Split the train data into train and validation data
train_size = int(0.8 * len(dummy_data))
val_size = len(dummy_data) - train_size
split_generator = torch.Generator().manual_seed(SEED)
train_subset_tmp, val_subset_tmp = random_split(
    dummy_data, [train_size, val_size], generator=split_generator
)

# Extract indices from them
train_idx = train_subset_tmp.indices
val_idx = val_subset_tmp.indices

# Create Subsets using the indices
train_dataset = datasets.Food101(
    root=DATA_PATH, split="train", download=False, transform=train_transform
)
val_dataset = datasets.Food101(
    root=DATA_PATH, split="train", download=False, transform=test_transform
)

# Create Subsets using the indices
train_subset = Subset(train_dataset, train_idx)
val_subset = Subset(val_dataset, val_idx)

# Download Test Dataset
test_dataset = datasets.Food101(
    root=DATA_PATH, split="test", download=True, transform=test_transform
)

In [ ]:
def seed_worker(worker_id):
    worker_seed = SEED + worker_id
    random.seed(worker_seed)
    np.random.seed(worker_seed)


train_generator = torch.Generator().manual_seed(SEED)
eval_generator = torch.Generator().manual_seed(SEED)

train_loader = DataLoader(
    train_subset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=1,
    pin_memory=True,
    worker_init_fn=seed_worker,
    generator=train_generator,
)

val_loader = DataLoader(
    val_subset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    worker_init_fn=seed_worker,
    generator=eval_generator,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    worker_init_fn=seed_worker,
    generator=eval_generator,
)

# 4. Model Architecture

![GoogLeNet](figures/GoogLeNet.png)
![GoogLeNet2](figures/GoogLeNet2.png)

1. Inception Modul

In [75]:
class Inception(nn.Module):
    def __init__(self, in_channels, out_1x1, re_3x3, out_3x3, re_5x5, out_5x5, out_pool):
        super().__init__()
        
        self.path1 = nn.Sequential(
            nn.Conv2d(in_channels, out_1x1, kernel_size=1, stride=1),
            nn.ReLU(inplace=True)
        )
        
        self.path2 = nn.Sequential(
            nn.Conv2d(in_channels, re_3x3, kernel_size=1,stride=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(re_3x3, out_3x3, kernel_size=3, stride=1, padding=1),
            nn.ReLU(inplace=True)
        )
        
        self.path3 = nn.Sequential(
            nn.Conv2d(in_channels, re_5x5, kernel_size=1, stride=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(re_5x5, out_5x5, kernel_size=5, stride=1, padding=2),
            nn.ReLU(inplace=True)
        )
        
        self.path4 = nn.Sequential(
            nn.MaxPool2d(kernel_size=3, stride=1, padding=1),
            nn.Conv2d(in_channels, out_pool, kernel_size=1, stride=1),
            nn.ReLU(inplace=True)
        )
        
    def forward(self, x):
        out1 = self.path1(x)
        out2 = self.path2(x)
        out3 = self.path3(x)
        out4 = self.path4(x)
        
        return torch.concat([out1, out2, out3, out4], dim=1)

2. Auxilary Classifier

In [76]:
class Aux(nn.Module):
    def __init__(self, in_channels, num_class):
        super().__init__()
        self.avg = nn.AvgPool2d(kernel_size=5, stride=3)
        self.conv = nn.Conv2d(in_channels, 128, kernel_size=1, stride=1)
        self.dropout = nn.Dropout(p=0.7)
        self.fc1 = nn.Linear(2048, 1024)
        self.relu = nn.ReLU(inplace=True)
        self.fc2 = nn.Linear(1024, num_class)
        
    def forward(self, x):
        x = self.avg(x)
        x = self.conv(x)
        x = self.relu(x)
        
        x = torch.flatten(x, 1)
        
        x = self.fc1(x)
        x = self.relu(x)
        x = self.dropout(x)
        x = self.fc2(x)
        
        return x
        

3. GoogLeNet

In [77]:
class GoogLeNet(nn.Module):
    def __init__(self, in_channels, num_class, dropout_rate):
        super().__init__()
        self.pre_layer = nn.Sequential(
            nn.Conv2d(in_channels, 64, kernel_size=7, stride=2, padding=3),
            nn.ReLU(inplace=True),
            
            nn.MaxPool2d(kernel_size=3, stride=2),
            nn.LocalResponseNorm(size=5, alpha=0.0001, beta=0.75, k=2.0),
            
            nn.Conv2d(64, 64, kernel_size=1, stride=1),
            nn.ReLU(inplace=True),
            
            nn.Conv2d(64, 192, kernel_size=3, stride=1, padding=1),
            nn.ReLU(inplace=True),
            
            nn.LocalResponseNorm(size=5, alpha=0.0001, beta=0.75, k=2.0)
        )
        
        self.a3 = Inception(in_channels=192, out_1x1=64, re_3x3=96, 
                        out_3x3=128, re_5x5=16, out_5x5=32, out_pool=32)
        
        self.b3 = Inception(in_channels=256, out_1x1=128, re_3x3=128, 
                        out_3x3=192, re_5x5=32, out_5x5=96, out_pool=64)
        
        self.a4 = Inception(in_channels=480, out_1x1=192, re_3x3=96,
                        out_3x3=208, re_5x5=16, out_5x5=48, out_pool=64)
        
        self.b4 = Inception(in_channels=512, out_1x1=160, re_3x3=112,
                        out_3x3=224, re_5x5=24, out_5x5=64, out_pool=64)
        
        self.c4 = Inception(in_channels=512, out_1x1=128, re_3x3=128,
                        out_3x3=256, re_5x5=24, out_5x5=64, out_pool=64)
        
        self.d4 = Inception(in_channels=512, out_1x1=112, re_3x3=144,
                        out_3x3=288, re_5x5=32, out_5x5=64, out_pool=64)
        
        self.e4 = Inception(in_channels=528, out_1x1=256, re_3x3=160,
                        out_3x3=320, re_5x5=32, out_5x5=128, out_pool=128)
        
        self.a5 = Inception(in_channels=832, out_1x1=256, re_3x3=160,
                        out_3x3=320, re_5x5=32, out_5x5=128, out_pool=128)
        
        self.b5 = Inception(in_channels=832, out_1x1=384, re_3x3=192,
                        out_3x3=384, re_5x5=48, out_5x5=128, out_pool=128)
        
        self.max_pool = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)
        self.avg_pool = nn.AvgPool2d(kernel_size=7, stride=1)
        self.fc = nn.Linear(1024, num_class)
        self.dropout = nn.Dropout(p=dropout_rate)
        
        self.aux1 = Aux(512, num_class)
        self.aux2 = Aux(528, num_class)

        self._initialize_weights()
        
    def forward(self, x):
        x = self.pre_layer(x)
        x = self.max_pool(x)
        
        x = self.a3(x)
        x = self.b3(x)
        x = self.max_pool(x)
        x = self.a4(x)
        
        aux1_out = None
        if self.training:
            aux1_out = self.aux1(x)
        
        x = self.b4(x)
        x = self.c4(x)
        x = self.d4(x)
        
        aux2_out = None
        if self.training:
            aux2_out = self.aux2(x)
        
        x = self.e4(x)
        x = self.max_pool(x)
        x = self.a5(x)
        x = self.b5(x)
        
        x = self.avg_pool(x)
        
        x = torch.flatten(x, 1)
        x = self.dropout(x)
        x = self.fc(x)
        
        if self.training:
            return x, aux1_out, aux2_out
        return x

    def _initialize_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d) or isinstance(m, nn.Linear):
                # Xavier/Glorot
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)

In [78]:
model = GoogLeNet(IN_CHANNELS, NUM_CLS, DROPOUT_RATE).to(device)
print(f"Total parameters: {(sum(p.numel() for p in model.parameters())/1e6):.2f}M")

# Use multiple GPUs if available
if torch.cuda.device_count() > 1:
    print(f"Using {torch.cuda.device_count()} GPUs")
    model = nn.DataParallel(model)

Total parameters: 10.61M


# 5. Training Preparation

1. Early Stopping

In [79]:
class EarlyStopping:
    def __init__(self, patience=10, delta=0, save_path='best_checkpoint.pth'):
        self.patience = patience
        self.delta = delta
        self.save_path = save_path
        
        self.best_loss = None
        self.verbose = False
        self.counter = 0
        self.early_stopp = False
        
    def __call__(self, val_loss, model):
        # For the first epoch
        if self.best_loss is None:
            self.save_checkpoint(val_loss, model)
        
        # If the loss didnt reduce as expect
        elif val_loss >= self.best_loss - self.delta:
            self.counter += 1
            print(f"Early Stopping counter: {self.counter} of of {self.patience}")
            if self.counter >= self.patience:
                self.early_stopp = True
        
        # The loss reduces enough  
        else:
            self.save_checkpoint(val_loss, model)
            self.counter = 0
        
    def save_checkpoint(self, val_loss, model):
        self.best_loss = val_loss
        if self.verbose:
            print('Saving best checkpoint ...')
        state_dict = model.module.state_dict() if hasattr(model, 'module') else model.state_dict()
        torch.save(state_dict, self.save_path)

2. Optimizer and Scheduler

In [80]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=LR, momentum=0.9, weight_decay=WEIGHT_DECAY)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=8, gamma=0.96)

scaler = torch.amp.GradScaler(device)

3. Train

In [81]:
def train(model, loader, criterion, optimizer, scheduler):
    model.train()
    train_loss, train_acc = 0, 0
    loop = tqdm(loader, desc="Training", leave=False)
    
    for x, y in loop:
        # Move training data to device
        x, y = x.to(device), y.to(device)
        
        # Clear gradients of last batch
        optimizer.zero_grad(set_to_none=True)
        
        # Get output and loss with Mixed Precision
        with torch.amp.autocast(device_type=device.type):
            main_out, aux1_out, aux2_out = model(x)
            main_loss = criterion(main_out, y)
            aux1_loss = criterion(aux1_out, y)
            aux2_loss = criterion(aux2_out, y)
            loss = main_loss + 0.3 * aux1_loss + 0.3 * aux2_loss
            
        # Scale up the loss to avoid Gradient Underflow and backpropagate
        scaler.scale(loss).backward()
        
        # Scale down for clipping and clip the gradient
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1)
        
        scaler.step(optimizer)
        
        # Update the scaler
        scaler.update()
        
        # Calculate total loss and acc
        train_loss += loss.item() * x.size(0)
        train_acc += (main_out.argmax(1) == y).sum().item()
        
    return train_loss / len(loader.dataset), train_acc / len(loader.dataset)

4. Validation

In [82]:
def validate(model, loader, criterion):
    model.eval()
    val_loss, val_acc = 0, 0
    loop = tqdm(loader, desc="Validation", leave=False)
    
    for x, y in loop:
        x, y = x.to(device), y.to(device)
        
        with torch.no_grad():
            out = model(x)
            loss = criterion(out, y)
            
        val_loss += loss.item() * x.size(0)
        val_acc += (out.argmax(1) == y).sum().item()
        
    return val_loss / len(loader.dataset), val_acc / len(loader.dataset)

5. Evaluation

In [83]:
def test(model, loader):
    model.eval()
    test_acc = 0
    loop = tqdm(loader, desc="Testing", leave=False)
    
    for x, y in loop:
        x, y = x.to(device), y.to(device)
        
        with torch.no_grad():
            out = model(x)

        test_acc += (out.argmax(1) == y).sum().item()
        
    return test_acc / len(loader.dataset)

# 6. Train

In [ ]:
wandb.init(
    project="GoogLeNet",
    config={
        "Architecture": "GoogLeNet",
        "epochs": EPOCHS,
        "batch_size": BATCH_SIZE,
        "image_size": IMG_SIZE
    }
)

train_accuracies, train_losses = [], []
val_accuracies, val_losses = [], []
early_stopping = EarlyStopping(patience=5)

for epoch in range(EPOCHS):
    train_loss, train_acc = train(model, train_loader, criterion,
                                  optimizer, scheduler)
    val_loss, val_acc = validate(model, val_loader, criterion)
    
    scheduler.step()
    
    train_accuracies.append(train_acc)
    val_accuracies.append(val_acc)
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    
    wandb.log({
        "lr": scheduler.get_last_lr(),
        "train_loss": train_loss,
        "val_loss": val_loss,
        "train_acc": train_acc,
        "val_acc": val_acc
    })
    
    print(f"Epoch {epoch+1}/{EPOCHS}: train_loss: {train_loss:.4f}, val_loss: {val_loss:.4f}, " +
          f"train_acc:  {train_acc:.4f}, test_acc: {val_acc:.4f}")
    
    early_stopping(val_loss, model)
    if early_stopping.early_stopp:
        print("Early Stopping")
        break
    
    # Stop at 10. Epoch because I dont want to run it too long
    # Since the script works and GoogLeNet is absolet already
    if epoch == 9:
        break
    
best_model = model.module if hasattr(model, 'module') else model
best_model.load_state_dict(torch.load('best_checkpoint.pth', map_location=device))
test_acc = test(model, test_loader)
print(f"Test accuracy: {test_acc:.4f}")
wandb.log(f"Test accuracy: {test_acc:.4f}")

wandb.finish()

train_acc,▁
train_loss,▁
val_acc,▁
val_loss,▁
train_acc,0.00995
train_loss,7.38442
val_acc,0.00957
val_loss,4.61554


Training:   0%|          | 0/474 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: avg_pool3d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


# 7. GradCAM

In [ ]:
!pip install grad-cam -q
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
import random

# Pick the last Inception Block to setup the GradCAM
# In GoogLeNet, feature maps are often taken from one of the last inception modules
target_layers = [model.b5]

cam = GradCAM(model=model, target_layers=target_layers)

# Pick a random image form the test set
imgs, labels = next(iter(test_loader))
index = random.randint(0, len(imgs) - 1)
input_tensor = imgs[index].unsqueeze(0).to(device)
label = labels[index].item()

# Generate the Grad-CAM heatmap
grayscale_cam = cam(input_tensor=input_tensor, targets=None)
grayscale_cam = grayscale_cam[0, :]

# Prepare the image for Visualization
# create a 1D Tensor from the tuple of per-channel means/std
# and reshape it to (3, 1, 1) because imgs[index] has (3, H, W)
mean = torch.tensor(stats[0]).view(3, 1, 1)
std = torch.tensor(stats[1]).view(3, 1, 1)
rgb_img = imgs[index] * std + mean # Denormalize
rgb_img = rgb_img.permute(1, 2, 0).numpy()
rgb_img = np.clip(rgb_img, 0, 1)

# Overlay the heatmap on the image
visual = show_cam_on_image(rgb_img, grayscale_cam, use_rgb=True)

# Get the Prediction
model.eval()
with torch.no_grad():
    out = model(input_tensor)
    pred = out.argmax(1).item()

# 7. Plotting
plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1)
plt.title(f"Original (Class: {label})")
plt.imshow(rgb_img)
plt.axis('off')
plt.subplot(1, 2, 2)
plt.title(f"Grad-CAM (Pred: {pred})")
plt.imshow(visual)
plt.axis('off')
plt.tight_layout()
plt.show()